# 03 — Run TEMPTED

Fit TEMPTED once to the full filtered cohort and save its subject, feature, and temporal loadings.

`format_tempted()` uses CLR for microbiome data. With `pseudo=NULL`, the package uses one-half of each sample's smallest non-zero value as its pseudocount. `threshold=1` disables additional feature filtering because notebook 02 already defines the retained genera.

**TEMPTED documentation:**  
https://rdrr.io/cran/tempted/man/format_tempted.html

In [1]:
suppressPackageStartupMessages(library(tempted)) # import tempted

# tempted settings
RANK <- 2
SMOOTH <- 1e-4
MAXITER <- 20
EPSILON <- 1e-4

root <- if (dir.exists("data")) "." else ".."
source <- tail(sort(list.dirs(file.path(root, "data", "preprocessing"), recursive = FALSE)), 1)
output <- file.path(root, "data", "tempted", format(Sys.time(), "%Y%m%d_%H%M%S"))
dir.create(output, recursive = TRUE)

# load the filtered counts and aligned metadata
counts <- read.csv(file.path(source, "counts_filtered.csv"), check.names = FALSE)
metadata <- read.csv(file.path(source, "metadata.csv"), stringsAsFactors = FALSE)

counts$sample_id <- NULL
counts[] <- lapply(counts, as.numeric)

In [2]:
# format the longitudinal data with tempted's clr preprocessing
x <- format_tempted(
  counts,
  metadata$age,
  metadata$subject_id,
  threshold = 1,
  pseudo = NULL,
  transform = "clr"
)

# centralize the longitudinal matrices
center <- svd_centralize(x, r = 1)

# fit the tempted model
model <- tempted(
  center$datlist,
  r = RANK,
  smooth = SMOOTH,
  maxiter = MAXITER,
  epsilon = EPSILON
)

Calculate the 1th Component

Convergence reached at dif=3.11966222358431e-05, iter=4

Calculate the 2th Component

Convergence reached at dif=9.93094079273926e-05, iter=6



In [3]:
# format subject component scores
subject_scores <- as.data.frame(model$A_hat)
names(subject_scores) <- paste0("component_", seq_len(ncol(subject_scores)))
subject_scores$subject_id <- rownames(subject_scores)

# format genus loadings
feature_loadings <- as.data.frame(model$B_hat)
names(feature_loadings) <- paste0("component_", seq_len(ncol(feature_loadings)))
feature_loadings$feature_id <- rownames(feature_loadings)

# format temporal loading
temporal_loadings <- as.data.frame(model$Phi_hat)
names(temporal_loadings) <- paste0("component_", seq_len(ncol(temporal_loadings)))
temporal_loadings$time <- model$time_Phi

In [4]:
# save tempted outputs
write.csv(subject_scores, file.path(output, "subject_scores.csv"), row.names = FALSE)
write.csv(feature_loadings, file.path(output, "feature_loadings.csv"), row.names = FALSE)
write.csv(temporal_loadings, file.path(output, "temporal_loadings.csv"), row.names = FALSE)
saveRDS(model, file.path(output, "model.rds"))

cat("Saved", RANK, "components to", output, "\n")

Saved 2 components to ../data/tempted/20260816_182040 
